# Example 1: [CII] Rotation Curve of a Confirmed High-z Rotator

**High-z Kinematic Corpus Z1 — EPS Research, Flynn D.C. (2026)**

This notebook demonstrates basic single-galaxy retrieval from the Z1 corpus JSON.  
We load J0817, the highest-velocity confirmed rotator in the corpus (V_rot,max = 252 km/s  
at z = 4.26), and reproduce its per-ring [CII] rotation curve with error bars, velocity  
dispersion profile, and dynamical mass estimate — all from a single JSON load.

**Corpus:** Flynn (2026), Zenodo DOI: [pending]  
**Source data:** Jones et al. (2021), MNRAS, arXiv:2104.03099 (Table C1)  
**Dependencies:** Python 3, numpy, matplotlib (standard library only)

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# ── Load corpus ───────────────────────────────────────────────────────────────
with open('high_z_kinematic_corpus_Z1.json') as f:
    corpus = json.load(f)



# Dwarf Example 02: DDO154 Omega Computation

**EPS Research — Dwarf/Irregular HI Corpus v1.0**

DDO154 is one of the best-studied dark-matter-dominated dwarfs.
This example reproduces the DDO154 omega computation from Flynn (2026):
omega = 6.86 rad/Gyr (SPARC rotmod) — consistent with the SPARC mean of 7.06.

The dwarf corpus median omega = 9.94 rad/Gyr (24 omega-ready LITTLE THINGS),
higher than the SPARC mean, consistent with deeper dark matter dominance.

**Corpus:** Flynn (2026), Zenodo DOI: 10.5281/zenodo.20320362  
**Sources:** LVHIS (Koribalski 2019), VLA-ANGST (Ott 2012), LITTLE THINGS (Oh 2015), WALLABY DR2  
**Dependencies:** Python 3, numpy, matplotlib

In [ ]:
# ── Retrieve J0817 ────────────────────────────────────────────────────────────
g = next(g for g in corpus['galaxies'] if g['galaxy'] == 'J0817')

print(f"Galaxy:          {g['galaxy']}")
print(f"Redshift:        z = {g['redshift']:.4f}")
print(f"Classification:  {g['class_jones2021']} (Jones et al. 2021)")
print(f"Quality tier:    {g['quality_tier']}")
print(f"Inclination:     {g['inc_kin_deg']} +/- {g['e_inc_kin_deg']} deg")
print(f"PA (kinematic):  {g['pa_kin_deg']} +/- {g['e_pa_kin_deg']} deg")
print(f"N rings:         {g['n_rings']}")
print(f"R range:         {g['r_min_kpc']:.2f} -- {g['r_max_kpc']:.2f} kpc")
print(f"Vrot max:        {g['vrot_max_kms']:.1f} km/s")
print(f"log Mdyn:        {g['log_mdyn_msun']:.3f} Msun")
print(f"log Mstar:       {g['log_mstar_msun']:.1f} Msun (Faisst et al. 2020)")
print(f"SFR:             {g['sfr_msun_yr']:.0f} Msun/yr")
print(f"Beam smeared:    {g['beam_smeared']} | corrected: {g['beam_smear_corrected']} (3DBarolo)")

# W15 disk criteria
w15 = g['w15_criteria']
print(f"\nWisnioski+2015 disk criteria: {w15['n_passed']}/5 passed")
for k, v in w15.items():
    if k != 'n_passed':
        print(f"  {k}: {v}")

In [ ]:
# ── Extract per-ring data ─────────────────────────────────────────────────────
d = g['data']

R       = np.array([p['R_kpc']       for p in d])
Vrot    = np.array([p['Vrot_kms']    for p in d])
eVrot   = np.array([p['e_Vrot_kms']  for p in d])
sigma   = np.array([p['sigma_kms']   for p in d])
esigma  = np.array([p['e_sigma_kms'] for p in d])
Mdyn    = np.array([p['Mdyn_msun']   for p in d])
vos     = np.array([p['v_over_sigma']for p in d])

print("Per-ring data:")
print(f"  {'R_kpc':>8}  {'Vrot':>8}  {'eVrot':>8}  {'sigma':>8}  {'esigma':>8}  {'v/sigma':>8}")
for i in range(len(R)):
    print(f"  {R[i]:8.2f}  {Vrot[i]:8.2f}  {eVrot[i]:8.2f}  {sigma[i]:8.2f}  {esigma[i]:8.2f}  {vos[i]:8.3f}")

In [ ]:
# ── Figure: rotation curve + dispersion + dynamical mass ─────────────────────
fig = plt.figure(figsize=(10, 8))
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.38, wspace=0.35)
ax1 = fig.add_subplot(gs[0, :])
ax2 = fig.add_subplot(gs[1, 0])
ax3 = fig.add_subplot(gs[1, 1])

# Panel 1: Rotation curve
ax1.errorbar(R, Vrot, yerr=eVrot, fmt='o-', color='#1f77b4',
             capsize=4, linewidth=1.8, markersize=7,
             label=r'$V_{\rm rot}$ (3DBarolo, beam-smear corrected)')
ax1.axhspan(0, 50, alpha=0.08, color='gray',
            label=r'$V_{\rm rot} < 50$ km/s (caution zone, not applicable here)')
ax1.set_xlabel('Radius (kpc)', fontsize=12)
ax1.set_ylabel(r'$V_{\rm rot}$ (km/s)', fontsize=12)
ax1.set_title(
    f'J0817 — ALPINE Confirmed Rotator, z = {g["redshift"]:.4f}\n'
    f'Loaded from High-z Kinematic Corpus Z1 (Flynn 2026)',
    fontsize=11)
ax1.legend(fontsize=9)
ax1.text(0.97, 0.08,
    f'inc = {g["inc_kin_deg"]}°  |  PA = {g["pa_kin_deg"]}°\n'
    f'log M$_{{\\rm dyn}}$ = {g["log_mdyn_msun"]:.3f} M$_\\odot$\n'
    f'W15: {w15["n_passed"]}/5 disk criteria',
    transform=ax1.transAxes, va='bottom', ha='right',
    fontsize=9, bbox=dict(boxstyle='round,pad=0.3', fc='white', alpha=0.7))
ax1.set_xlim(0, R.max() * 1.2)
ax1.set_ylim(0, Vrot.max() * 1.35)

# Panel 2: Velocity dispersion
ax2.errorbar(R, sigma, yerr=esigma, fmt='s-', color='#d62728',
             capsize=4, linewidth=1.5, markersize=6,
             label=r'$\sigma_{\rm mean}$')
ax2.set_xlabel('Radius (kpc)', fontsize=11)
ax2.set_ylabel(r'$\sigma$ (km/s)', fontsize=11)
ax2.set_title('Velocity Dispersion Profile', fontsize=10)
ax2.legend(fontsize=9)
ax2.set_xlim(0, R.max() * 1.2)

# Panel 3: v/sigma per ring
ax3.plot(R, vos, 'D-', color='#2ca02c', linewidth=1.5, markersize=6,
         label=r'$V_{\rm rot}/\sigma$ per ring')
ax3.axhline(1.0, color='gray', linestyle='--', linewidth=1,
            label=r'$V/\sigma = 1$ (dispersion boundary)')
ax3.set_xlabel('Radius (kpc)', fontsize=11)
ax3.set_ylabel(r'$V_{\rm rot}/\sigma$', fontsize=11)
ax3.set_title(r'Kinematic State $V/\sigma$', fontsize=10)
ax3.legend(fontsize=9)
ax3.set_xlim(0, R.max() * 1.2)

plt.savefig('fig_hz_nb1_j0817_rc.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig_hz_nb1_j0817_rc.png')

In [ ]:
# ── Summary table: all 8 tier-1 rotators ─────────────────────────────────────
rotators = [g for g in corpus['galaxies']
            if g.get('is_rotator') and g.get('quality_tier') == 1]

print(f"{'Galaxy':<18} {'z':>6} {'n_rings':>7} {'r_max':>7} "
      f"{'Vrot_out':>9} {'eVrot':>7} {'sigma_out':>10} {'v/sig':>7} {'logMdyn':>8} {'W15':>4}")
print('-' * 90)

for g in sorted(rotators, key=lambda x: x['redshift']):
    last = g['data'][-1]
    print(f"{g['galaxy']:<18} {g['redshift']:>6.4f} {g['n_rings']:>7d} "
          f"{g['r_max_kpc']:>7.2f} "
          f"{last['Vrot_kms']:>9.2f} {last['e_Vrot_kms']:>7.2f} "
          f"{last['sigma_kms']:>10.2f} {last['v_over_sigma']:>7.3f} "
          f"{g['log_mdyn_msun']:>8.3f} {g['w15_criteria']['n_passed']:>4d}/5")

print()
vrot_vals = [g['data'][-1]['Vrot_kms'] for g in rotators]
mdyn_vals = [g['log_mdyn_msun'] for g in rotators]
print(f"Vrot (outermost): {min(vrot_vals):.1f} -- {max(vrot_vals):.1f} km/s")
print(f"log Mdyn range:   {min(mdyn_vals):.3f} -- {max(mdyn_vals):.3f} Msun")
print(f"\nNote: inclination uncertainty not propagated into Vrot errors")
print(f"(3DBarolo systematic; see Jones et al. 2021 Section 5.3.2).")
print(f"All rotators: beam_smear_corrected = True (3DBarolo mitigates but")
print(f"does not eliminate beam-smearing at ~6-7 kpc ALMA resolution).")